In [ ]:
!pip install -r requirements_vit5.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/drive/MyDrive/demo/requirements_vit5.txt'


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())


PyTorch: 2.13.0+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1


Test on base ViT5

In [ ]:
from datasets import load_dataset
import torch
import json
from jiwer import wer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# ============================================================
# 1. Load BASE ViT5
# ============================================================

MODEL_NAME = "VietAI/vit5-base"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

print("Device:", device)
print("Model: Base VietAI/vit5-base")


# ============================================================
# 2. Test dataset
# ============================================================

test_dataset = load_dataset("csv", data_files="\data\rq1\ec\output_phoasr_test.csv") #insert test files here
test_dataset = test_dataset["train"]

print("Test examples:", len(test_dataset))


# ============================================================
# 3. Evaluate
# ============================================================

asr_wers = []
vit5_wers = []

improved = 0
unchanged = 0
worsened = 0

results = []


for i, example in enumerate(test_dataset):

    asr_text = example["asr_text"]
    reference = example["reference_text"]

    prompt = (
        "correct the transcription: "
        + asr_text
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024, #Note: this is the maximum number of tokens the model will generate. For this dataset there were very little number of transcriptions that went over this limit.
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # WER
    # --------------------------------------------------------

    asr_wer = wer(
        reference,
        asr_text
    )

    vit5_wer = wer(
        reference,
        prediction
    )

    asr_wers.append(asr_wer)
    vit5_wers.append(vit5_wer)

    # --------------------------------------------------------
    # Categorize
    # --------------------------------------------------------

    if vit5_wer < asr_wer:
        result = "improved"
        improved += 1

    elif vit5_wer > asr_wer:
        result = "worsened"
        worsened += 1

    else:
        result = "unchanged"
        unchanged += 1

    # --------------------------------------------------------
    # Save individual result
    # --------------------------------------------------------

    results.append({
        "index": i,
        "asr_text": asr_text,
        "vit5_prediction": prediction,
        "reference_text": reference,
        "asr_wer": asr_wer,
        "vit5_wer": vit5_wer,
        "result": result
    })

    print(
        f"{i + 1}/{len(test_dataset)} | "
        f"ASR WER: {asr_wer:.4f} | "
        f"Base ViT5 WER: {vit5_wer:.4f} | "
        f"{result}"
    )


# ============================================================
# 4. Overall results
# ============================================================

mean_asr_wer = (
    sum(asr_wers) / len(asr_wers)
)

mean_vit5_wer = (
    sum(vit5_wers) / len(vit5_wers)
)

absolute_reduction = (
    mean_asr_wer - mean_vit5_wer
)

relative_reduction = (
    absolute_reduction / mean_asr_wer
    if mean_asr_wer > 0
    else 0
)


summary = {
    "model": "VietAI/vit5-base",
    "dataset": "test",
    "num_examples": len(test_dataset),

    "mean_asr_wer": mean_asr_wer,
    "mean_vit5_wer": mean_vit5_wer,

    "absolute_wer_reduction": absolute_reduction,
    "relative_wer_reduction": relative_reduction,

    "improved": improved,
    "unchanged": unchanged,
    "worsened": worsened
}


# ============================================================
# 5. Save JSON
# ============================================================

output = {
    "results": results,
    "summary": summary
}

JSON_PATH = (
    "data\rq1\ec\output\base_vit5_phoasr_test_results.json"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 6. Print summary
# ============================================================

print()
print("=" * 80)
print("BASE ViT5 — TEST RESULTS")
print("=" * 80)

print(f"Examples:               {len(test_dataset)}")
print(f"Mean ASR WER:           {mean_asr_wer:.4f}")
print(f"Mean Base ViT5 WER:     {mean_vit5_wer:.4f}")
print(f"Absolute WER reduction: {absolute_reduction:.4f}")
print(
    f"Relative WER reduction: "
    f"{relative_reduction * 100:.2f}%"
)

print()
print(f"Improved:               {improved}")
print(f"Unchanged:              {unchanged}")
print(f"Worsened:               {worsened}")

print()
print(f"Saved to: {JSON_PATH}")

Device: cpu
Model: Base VietAI/vit5-base


Generating train split: 0 examples [00:00, ? examples/s]

Test examples: 557
1/557 | ASR WER: 0.2568 | Base ViT5 WER: 0.9662 | worsened
2/557 | ASR WER: 0.1547 | Base ViT5 WER: 0.8567 | worsened
3/557 | ASR WER: 0.0733 | Base ViT5 WER: 1.0267 | worsened
4/557 | ASR WER: 0.1803 | Base ViT5 WER: 1.1967 | worsened
5/557 | ASR WER: 0.2806 | Base ViT5 WER: 0.9235 | worsened
6/557 | ASR WER: 0.1645 | Base ViT5 WER: 0.9013 | worsened
7/557 | ASR WER: 0.4645 | Base ViT5 WER: 1.2796 | worsened
8/557 | ASR WER: 0.2561 | Base ViT5 WER: 5.3110 | worsened
9/557 | ASR WER: 0.6471 | Base ViT5 WER: 1.7206 | worsened
10/557 | ASR WER: 0.2614 | Base ViT5 WER: 0.9477 | worsened
11/557 | ASR WER: 0.2766 | Base ViT5 WER: 1.3972 | worsened
12/557 | ASR WER: 0.0645 | Base ViT5 WER: 5.3065 | worsened
13/557 | ASR WER: 0.2051 | Base ViT5 WER: 1.1966 | worsened
14/557 | ASR WER: 0.1450 | Base ViT5 WER: 5.9924 | worsened
15/557 | ASR WER: 0.3409 | Base ViT5 WER: 1.5625 | worsened
16/557 | ASR WER: 0.1298 | Base ViT5 WER: 5.0763 | worsened
17/557 | ASR WER: 0.1138 | Bas

Test on fine-tuned ViT5

training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5-asr-correction",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=False,
    fp16=True,
    seed=42,
    data_seed=42,
    report_to="none",
)

In [ ]:
from datasets import load_dataset
import torch
import json
from jiwer import wer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# ============================================================
# 1. Paths
# ============================================================

MODEL_WEIGHTS = (
    "vit5-asr-correction/final.pth"
)

TOKENIZER_PATH = (
    "vit5-asr-correction/final"
)

# ============================================================
# 2. Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 3. Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH,
    use_fast=False
)


# ============================================================
# 4. Recreate base ViT5 architecture
# ============================================================

model = AutoModelForSeq2SeqLM.from_pretrained(
    "VietAI/vit5-base"
)


# ============================================================
# 5. Load your fine-tuned weights
# ============================================================

state_dict = torch.load(
    MODEL_WEIGHTS,
    map_location="cpu"
)

model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Fine-tuned ViT5 loaded successfully.")

In [ ]:
# ============================================================
# 2. Test dataset
# ============================================================

test_dataset = load_dataset("csv", data_files="\data\results\rq1\output_phoasr_test.csv") #insert test file here
test_dataset = test_dataset["train"]

print("Test examples:", len(test_dataset))


# ============================================================
# 3. Evaluate
# ============================================================

asr_wers = []
vit5_wers = []

improved = 0
unchanged = 0
worsened = 0

results = []


for i, example in enumerate(test_dataset):

    asr_text = example["asr_text"]
    reference = example["reference_text"]

    prompt = (
        "correct the transcription: "
        + asr_text
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # WER
    # --------------------------------------------------------

    asr_wer = wer(
        reference,
        asr_text
    )

    vit5_wer = wer(
        reference,
        prediction
    )

    asr_wers.append(asr_wer)
    vit5_wers.append(vit5_wer)

    # --------------------------------------------------------
    # Categorize
    # --------------------------------------------------------

    if vit5_wer < asr_wer:
        result = "improved"
        improved += 1

    elif vit5_wer > asr_wer:
        result = "worsened"
        worsened += 1

    else:
        result = "unchanged"
        unchanged += 1

    # --------------------------------------------------------
    # Save individual result
    # --------------------------------------------------------

    results.append({
        "index": i,
        "asr_text": asr_text,
        "vit5_prediction": prediction,
        "reference_text": reference,
        "asr_wer": asr_wer,
        "vit5_wer": vit5_wer,
        "result": result
    })

    print(
        f"{i + 1}/{len(test_dataset)} | "
        f"ASR WER: {asr_wer:.4f} | "
        f"Base ViT5 WER: {vit5_wer:.4f} | "
        f"{result}"
    )


# ============================================================
# 4. Overall results
# ============================================================

mean_asr_wer = (
    sum(asr_wers) / len(asr_wers)
)

mean_vit5_wer = (
    sum(vit5_wers) / len(vit5_wers)
)

absolute_reduction = (
    mean_asr_wer - mean_vit5_wer
)

relative_reduction = (
    absolute_reduction / mean_asr_wer
    if mean_asr_wer > 0
    else 0
)


summary = {
    "model": "VietAI/vit5-base",
    "dataset": "test",
    "num_examples": len(test_dataset),

    "mean_asr_wer": mean_asr_wer,
    "mean_vit5_wer": mean_vit5_wer,

    "absolute_wer_reduction": absolute_reduction,
    "relative_wer_reduction": relative_reduction,

    "improved": improved,
    "unchanged": unchanged,
    "worsened": worsened
}


# ============================================================
# 5. Save JSON
# ============================================================

output = {
    "results": results,
    "summary": summary
}

JSON_PATH = (
    "data\rq1\ec\output\ft_vit5_phoasr_test_results.json"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 6. Print summary
# ============================================================

print()
print("=" * 80)
print("BASE ViT5 — TEST RESULTS")
print("=" * 80)

print(f"Examples:               {len(test_dataset)}")
print(f"Mean ASR WER:           {mean_asr_wer:.4f}")
print(f"Mean Base ViT5 WER:     {mean_vit5_wer:.4f}")
print(f"Absolute WER reduction: {absolute_reduction:.4f}")
print(
    f"Relative WER reduction: "
    f"{relative_reduction * 100:.2f}%"
)

print()
print(f"Improved:               {improved}")
print(f"Unchanged:              {unchanged}")
print(f"Worsened:               {worsened}")

print()
print(f"Saved to: {JSON_PATH}")